## Create Datasets and Dataloaders (<code>data_setup.py</code>)

Let's write a script for creating datasets and dataloaders for the image classificating problem

We are going to convert useful <code>Dataset</code> and <code>Dataloaders</code> by creating a function called <code>create_dataloaders()</code> and we are going to write in the python script <code>datasetup.py</code> using the one line code <code>%%writefile going_modular/data_setup.py</code>.

In [31]:
%%writefile going_modular/data_setup.py
"""
Contains functionalities for creating PyTorch Dataloaders for image classification data.
"""
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS=0

def create_train_dataloaders(
    train_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=0):
    '''
    Creates training and testing Dataloaders.

    Takes in a training directory and testing directory path and turns them into PyTorch Datasets and PyTorch Dataloaders.

    Args:
        train_dir: Path to the training directory.
        test_dir: Path to the testing directory.
        transform: torchvision transforms to perform on training and testing data.
        batch_size: Number of samples in each of the Dataloaders.
        num_workers: An integer for number of workers per Dataloder.

    Returns:
        A tuple of (train_dataloader, test_dataloader, class_names).
        Where class_names is a list of the target classes.   
    '''

    # Use Imagefolder to create datasets
    train_data=datasets.ImageFolder(train_dir,transform=transform)
    
    #Get classes
    class_names=train_data.classes

    #Turn images into dataloaders
    train_dataloader=DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True)

    

    return train_dataloader,  class_names


def create_test_dataloaders(test_dir: str,
                           transform: transforms.Compose,
                            batch_size: int,
                            num_workers: int=0):
    
    test_data=datasets.ImageFolder(test_dir,transform=transform)

    test_dataloader=DataLoader(
        test_data,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=True)

    return test_dataloader

if __name__ == "__main__":
    train_loop()


Overwriting going_modular/data_setup.py


# Making a model (<code>model_builder.py</code>)

### Creating <code>train_step()</code>,<code>test_step()</code> and function <code>train</code> to combine them

* <code>train_step()</code>- takes in a model, a <code>Dataloader</code>, a loss function and an optimizer and trains the model on the dataloader.

* <code>test_step()</code>- takes in a model, a <code>Dataloader</code>, a loss function and evaluates the model on the <code>Dataloader</code>

* <code>train()</code>- performs the above two steps together for a given number of epochs and returns a results dictionary.

In [35]:
%%writefile going_modular/engine.py
"""
Contains functions for training and testing PyTorch model.
"""
import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device: torch.device)-> Tuple[float,float]:
    """Trains a PyTorch model for a single epoch.
    Turns a target PyTorch model to training mode and then runs through all of the required training steps (forward pass, loss calculation, optimizer step).

    Args:
        model: A PyTorch model to be trained.
        dataloader: A Dataloader instance for the model to be trained on.
        loss_fn: A PyTorch loss function to minimalize.
        optimizer: A PyTorch optimizer to help minimalize the loss function.
        device: A target device to compute on (e.g. "cuda" or "cpu")

    Returns:
        A tuple of training loss and training accuracy metrics.
        In the form (train_loss, train_accuracy).

    """
    #Put model in train mode
    model.train()

    #Setup train loss and train accuracy values
    train_loss, train_acc=0,0

    #Loop through data loader , data batches
    for batch, (X,y) in enumerate(dataloader):

        #Send to GPU 
        X,y=X.to(device), y.to(device)

        # 1. Do the forward pass
        y_pred=model(X)
        
        # 2. Calculate the loss
        loss=loss_fn(y_pred,y)
        train_loss+=loss.item()

        # 3. Optmizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer zero grad
        optimizer.step()

        # Calculate and accumulate all accuracy metrics across all batches
        y_pred_class=torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
        train_acc+=(y_pred_class==y).sum().item()/len(y_pred)

    # Adjust metrics and accuracy per batch
    train_loss=train_loss/len(dataloader)
    train_acc=train_acc/len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """
    Tests a PyTorch model for a single epoch.

    Turns a PyTorch model to "eval" mode and then performs a forward pass on a testing dataset.

    Args:
        model: A PyTorch model for a dataset
        dataloader: A Dataset instance to test the model on
        loss_fn: A PyTorch loss function to calculate the loss on the test dataset
        device: A target device to compute

    Returns:
        A tuple of testing loss and testing accuracy metrics.
        In the form of (testing loss) and (testing accuracy)
    """

    
    # Put model in eval
    model.eval()

    # Define the test loss and test accuracy
    test_loss, test_accuracy = 0, 0

    # Turn on the inference mode
    with torch.inference_mode():
        # Iterating through the dataloader 
        for batch, (X, y) in enumerate(dataloader):
            # Send the computation to GPU
            X, y = X.to(device), y.to(device)

            # 1. Doing the forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy - FIXED HERE
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_accuracy += ((test_pred_labels == y).sum().item() / len(y))
    
    # Normalize by number of batches - ADDED THIS NORMALIZATION
    test_loss = test_loss / len(dataloader)
    test_accuracy = test_accuracy / len(dataloader)
    
    return test_loss, test_accuracy

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device)-> Dict[str, list]:
    
    """The function does the train step for all the batches and then tests for all the batch for the given epochs by using the 
    train_step() and test_step() functions with the given loss function, an optimizer.

    Calculates, prints and stores the values of the accuracy metrics

    Args:
        model: A PyTorch model to train and test
        train_dataloader: A dataloader instance to train the dataset
        test_dataloader: A dataloader instance to test the dataset
        optimizer: A PyTorch opimizer to help minimize the loss funciton
        loss_fn: A function that calculates the loss in the predicted and the actual output on both the datasets.
        epochs: An integer indicating how many epochs to train for.
        device: A target device to compute on ("cpu" or "gpu")

    Returns:
        A dictionary of training and testing loss as well as training and testing accuracy metrics. Each of metrics has a value in the list
         for each epoch.
        In the form: {train_loss:[....],
                      test_loss:[....],
                      train_acc:[....],
                      test_acc:[....]}
    
    """
    #Creating empty dictionary
    results={"train_loss":[],
             "test_loss":[],
             "train_acc":[],
             "test_acc":[]}

    #Loop through each epoch and perform train_step() and test_step()
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc=train_step(model=model,
                                         dataloader=train_dataloader,
                                         loss_fn=loss_fn,
                                         device=device,
                                         optimizer=optimizer)

        test_loss, test_acc=test_step(model=model,
                                      dataloader=test_dataloader,
                                      loss_fn=loss_fn,
                                      device=device)

        #Put the values in the results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        #Print out what's happening
        print(f"for epoch={epoch+1}, train-loss={train_loss:.4f} | test-loss={test_loss:.4f} | train-accuracy={train_acc:.4f} |  test-accuracy={test_acc:.4f}")
    return results


if __name__ == "__main__":
    train_loop()


Overwriting going_modular/engine.py


In [42]:
%%writefile going_modular/utils_save_3.py
"""Contains various utility functions for PyTorch model training and saving."""
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
    """
    Saves a PyTorch model to a directory.

    Args:
        model (torch.nn.Module): The PyTorch model to save.
        target_dir (str): The directory where the model will be saved.
        model_name (str): The filename for the saved model (must end with '.pth' or '.pt').
    """

    # Create target directory if it doesn't exist
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    # Validate model filename
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), \
        "model_name should end with '.pt' or '.pth'"

    # Define model save path
    model_save_path = target_dir_path / model_name

    # Save the model state_dict()
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(), f=model_save_path)


Writing going_modular/utils_save_3.py
